# Crosswalks of Italy — the notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/crosswalks-italy/story_001_crosswalks.ipynb)

This notebook reproduces **every figure and statistic** in
[Crosswalks of Italy](https://pointzerofive.net/2026/02/crosswalks-italy),
story #001 on *pointzerofive*. Run it top to bottom (`Runtime → Run all` in Colab);
it fetches the published dataset straight from the story repository.

**The question:** how safe is crossing the street in Italy — and does it depend on where you live?

In [ ]:
import json
import math
import pathlib
import urllib.request

import matplotlib.pyplot as plt
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/crosswalks-italy/crosswalks-italy.json"
LOCAL_PATH = next((p for p in (pathlib.Path("crosswalks-italy.json"), pathlib.Path("../data/crosswalks-italy.json")) if p.exists()), None)

if LOCAL_PATH:  # running inside a checkout
    raw = json.loads(LOCAL_PATH.read_text())
else:  # running in Colab / anywhere else
    with urllib.request.urlopen(DATA_URL) as response:
        raw = json.loads(response.read())

cities = pd.DataFrame(raw["cities"])
print(f"{len(cities)} provincial capitals, {raw['summary']['total_crosswalks']:,} mapped crossings")
cities

## The measure

One score per city: **crossings per 10 000 inhabitants** — pedestrian crossings from
OpenStreetMap (`highway=crossing` nodes inside municipal boundaries) divided by ISTAT
resident population. Per capita, not raw counts, so a city of 60 000 and a city of
2.8 million are comparable.

## The finding: north vs south

We classify each city into the standard ISTAT macro-areas (Nord / Centro / Mezzogiorno)
and compare group means.

In [ ]:
SOUTH = {"Puglia", "Campania", "Sicilia", "Calabria", "Basilicata", "Molise", "Abruzzo", "Sardegna"}
CENTER = {"Toscana", "Lazio", "Umbria", "Marche"}
cities["area"] = cities["region"].map(
    lambda r: "south" if r in SOUTH else ("center" if r in CENTER else "north")
)
cities.groupby("area")["score"].agg(["count", "mean"]).round(1)

In [ ]:
AREA_COLOR = {"north": "#444444", "center": "#999999", "south": "#c0392b"}
plot_df = cities.sort_values("score")

fig, ax = plt.subplots(figsize=(7, 5))
ax.hlines(plot_df["name"], 0, plot_df["score"], color="#dddddd", linewidth=1)
ax.scatter(plot_df["score"], plot_df["name"], c=plot_df["area"].map(AREA_COLOR), s=55, zorder=3)
for area in ("north", "south"):
    ax.axvline(
        cities.loc[cities["area"] == area, "score"].mean(),
        color=AREA_COLOR[area], linestyle="--", linewidth=1, alpha=0.7,
        label=f"{area} mean",
    )
ax.set_xlabel("crossings per 10 000 inhabitants")
ax.set_title("Figure 1 — the two groups do not overlap", loc="left", fontsize=11)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## But… with only 14 cities, couldn't this be luck?

Fair question — and at n = 14 a t-test would be fragile. So we use an **exact permutation
argument** instead, which needs no distributional assumptions at all.

Under the null hypothesis (macro-area has nothing to do with crosswalk provision), every
arrangement of the 14 cities across the ranking is equally likely. The number of ways to
choose which 6 cities occupy the bottom 6 ranks is C(14, 6) — and exactly **one** of those
arrangements puts all six southern cities there.

In [ ]:
n, n_south = len(cities), int((cities["area"] == "south").sum())
bottom = set(cities.nsmallest(n_south, "score")["name"])
south_names = set(cities.loc[cities["area"] == "south", "name"])

print(f"All {n_south} southern cities occupy the bottom {n_south} ranks: {bottom == south_names}")
p_exact = 1 / math.comb(n, n_south)
print(f"Exact permutation p = 1/{math.comb(n, n_south):,} = {p_exact:.5f} ({p_exact:.3%})")

Random luck is not a plausible explanation.

## But… is it just city size?

Also checkable: if big cities simply couldn't keep up, population should predict the score.

In [ ]:
r_pop = cities["population"].corr(cities["score"])
print(f"r(population, score) = {r_pop:.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(cities["population"] / 1e6, cities["score"],
           c=cities["area"].map(AREA_COLOR), s=55)
ax.set_xlabel("population (millions)")
ax.set_ylabel("crossings per 10 000 inhabitants")
ax.set_title(f"Figure 2 — size explains nothing (r = {r_pop:.2f})", loc="left", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

The best city (Bolzano) and the worst (Crotone) are both small. Neither chance nor city
size explains the gap — what remains is sustained local investment and planning tradition.
**Infrastructure is policy.**

---

## Data and methods

- **Crossings:** OpenStreetMap, `highway=crossing` nodes within municipal boundaries.
  Map data © [OpenStreetMap contributors](https://www.openstreetmap.org/copyright),
  available under the Open Database License (ODbL).
- **Population:** ISTAT, 2023.
- **Sample:** 14 provincial capitals selected for pipeline coverage — *not* a random sample
  of all 107 capitals; the permutation result is exact conditional on this sample.
- **Known threat to validity:** OSM mapping completeness may itself vary by region.
- Extraction pipeline: [`crosswalks_pipeline.py`](https://github.com/john-bruene/pointzerofive-data/blob/main/crosswalks-italy/crosswalks_pipeline.py)

*Story: [pointzerofive.net/2026/02/crosswalks-italy](https://pointzerofive.net/2026/02/crosswalks-italy)*